In [1]:
# 1. Instalar as bibliotecas necessárias no ambiente do Colab
!pip install -q -U google-genai pandas tabulate

# 2. Importar as bibliotecas e carregar a chave a partir dos Secrets
import pandas as pd
from google.colab import userdata
from google import genai

# Carregar a chave guardada nos Secrets do Colab
api_key = userdata.get('GEMINI_API_KEY')

# Inicializar o cliente do Gemini
client = genai.Client(api_key=api_key)

print("Ambiente configurado e pronto a utilizar!")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.0 which is incompatible.


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Ambiente configurado e pronto a utilizar!


In [2]:
import pandas as pd

# 1. Definir os dados de vendas
dados = {
    'Data': ['2026-10-01', '2026-10-01', '2026-10-02', '2026-10-03', '2026-10-04', '2026-10-05', '2026-10-05'],
    'Produto': ['Notebook Pro', 'Mouse Sem Fio', 'Cadeira Ergonómica', 'Teclado Mecânico', 'Notebook Pro', 'Monitor 4K', 'Mouse Sem Fio'],
    'Categoria': ['Eletrónicos', 'Acessórios', 'Móveis', 'Acessórios', 'Eletrónicos', 'Eletrónicos', 'Acessórios'],
    'Quantidade': [2, 15, 3, 8, 1, 4, 10],
    'Valor_Unitario': [4500.00, 80.00, 1200.00, 250.00, 4500.00, 2200.00, 80.00],
    'Total': [9000.00, 1200.00, 3600.00, 2000.00, 4500.00, 8800.00, 800.00]
}

# 2. Guardar os dados num ficheiro CSV
df_exemplo = pd.DataFrame(dados)
df_exemplo.to_csv('vendas_exemplo.csv', index=False)

print("Ficheiro 'vendas_exemplo.csv' criado com sucesso!")

Ficheiro 'vendas_exemplo.csv' criado com sucesso!


In [2]:
import pandas as pd
from google import genai
from google.colab import userdata

# 1. Ler o arquivo CSV criado no Passo 3
df = pd.read_csv("vendas_exemplo.csv")
tabela_markdown = df.to_markdown(index=False)

# 2. Inicializar o cliente com a chave dos Secrets
api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

# 3. Construir o prompt
prompt = f"""
Você é um analista de dados executivo.
Analise os dados de vendas abaixo e crie um relatório executivo estruturado.

DADOS DE VENDAS:
{tabela_markdown}

ESTRUTURA DESEJADA DO RELATÓRIO:
- **Resumo Geral:** Total faturado e volume total de itens vendidos.
- **Destaques de Vendas:** Produtos e categorias líderes em receita.
- **Pontos de Atenção:** Oportunidades de melhoria ou concentração de vendas em poucos produtos.
- **Recomendações Práticas:** 2 a 3 ações recomendadas para a liderança.

Seja direto, profissional e use marcadores (bullet points).
"""

# 4. Enviar a solicitação com tentativa automática em modelos alternativos caso ocorra 503
modelos_tentativa = ["gemini-3.8-flash", "gemini-3.5-flash", "gemini-3.7-flash"]
relatorio = None

for modelo in modelos_tentativa:
    try:
        print(f"Solicitando análise com o modelo: {modelo}...")
        response = client.models.generate_content(
            model=modelo,
            contents=prompt,
        )
        relatorio = response.text
        print(f"Sucesso ao gerar com o modelo {modelo}!\n")
        break
    except Exception as e:
        print(f"O modelo {modelo} está em alta demanda. Tentando o próximo modelo...")

# 5. Exibir e salvar o relatório
if relatorio:
    print("=== RELATÓRIO EXECUTIVO GERADO PELO GEMINI ===\n")
    print(relatorio)

    with open("relatorio_executivo.md", "w", encoding="utf-8") as f:
        f.write(relatorio)

    print("\nRelatório salvo com sucesso como 'relatorio_executivo.md' na pasta de arquivos!")
else:
    print("Não foi possível conectar aos modelos neste segundo. Aguarde 10 segundos e tente rodar novamente.")

Solicitando análise com o modelo: gemini-3.8-flash...
O modelo gemini-3.8-flash está em alta demanda. Tentando o próximo modelo...
Solicitando análise com o modelo: gemini-3.5-flash...
Sucesso ao gerar com o modelo gemini-3.5-flash!

=== RELATÓRIO EXECUTIVO GERADO PELO GEMINI ===

Aqui está o relatório executivo estruturado com base na análise dos dados de vendas do período (01 a 05 de outubro de 2026):

---

# 📊 Relatório Executivo de Vendas

### **1. Resumo Geral**
* **Faturamento Total:** R$ 29.900,00
* **Volume Total de Itens Vendidos:** 43 unidades
* **Ticket Médio por Item:** R$ 695,35
* **Período Analisado:** 01/10/2026 a 05/10/2026 (5 dias)

---

### **2. Destaques de Vendas**
* **Categoria Líder em Receita:** **Eletrónicos** movimentou **R$ 22.300,00**, representando **74,6%** de todo o faturamento no período.
* **Produto Campeão de Faturamento:** **Notebook Pro** gerou **R$ 13.500,00** (45,2% da receita total) com apenas 3 unidades vendidas.
* **Líder em Volume de Vendas:** *

In [6]:
# 1. Autenticar a sua conta do Google no Colab
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
import pandas as pd

# 2. Conectar com o serviço do Google Sheets
creds, _ = default()
gc = gspread.authorize(creds)

# 3. Inserir a URL da sua planilha
url_planilha = "https://docs.google.com/spreadsheets/d/1L2t-uXjbEiWQ_b_y6rdpXMBPN_oHXQfshCc1UvZTpWk/edit?gid=0#gid=0"

# 4. Abrir a primeira aba da planilha e carregar os dados
aba = gc.open_by_url(url_planilha).sheet1
dados = aba.get_all_records()

# 5. Converter em um DataFrame do Pandas
df = pd.DataFrame(dados)

print("Dados da planilha carregados com sucesso no Colab!\n")
print(df)

MessageError: Error: credential propagation was unsuccessful

In [9]:
import pandas as pd

# 1. Cole aqui o URL completo da sua planilha do Google Sheets
url_planilha = "https://docs.google.com/spreadsheets/d/1L2t-uXjbEiWQ_b_y6rdpXMBPN_oHXQfshCc1UvZTpWk/edit?gid=0#gid=0"

# 2. Extrair o ID da planilha e gerar o link de leitura CSV direta
sheet_id = url_planilha.split('/d/')[1].split('/')[0]
csv_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"

# 3. Carregar os dados diretamente em tempo real
df = pd.read_csv(csv_url)

print("Dados da planilha carregados com sucesso no Colab!\n")
print(df)

Dados da planilha carregados com sucesso no Colab!

         Data             Produto    Categoria  Quantidade  Valor_Unitario  \
0  2026-10-01        Notebook Pro  Eletrônicos           2          4500.0   
1  2026-10-01       Mouse Sem Fio   Acessórios          15            80.0   
2  2026-10-02  Cadeira Ergonômica       Móveis           3          1200.0   
3  2026-10-03    Teclado Mecânico   Acessórios           8           250.0   
4  2026-10-04        Notebook Pro  Eletrônicos           1          4500.0   
5  2026-10-05          Monitor 4K  Eletrônicos           4          2200.0   
6  2026-10-05       Mouse Sem Fio   Acessórios          10            80.0   
7  2026-10-06        Notebook Pro  Eletrônicos           3          4500.0   

     Total  
0   9000.0  
1   1200.0  
2   3600.0  
3   2000.0  
4   4500.0  
5   8800.0  
6    800.0  
7  13500.0  


In [11]:
import pandas as pd
from google import genai
from google.colab import userdata

# 1. Converter a tabela da planilha (carregada no Passo 2) para formato Markdown
tabela_markdown = df.to_markdown(index=False)

# 2. Inicializar o cliente do Gemini
api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

# 3. Criar o prompt de análise
prompt = f"""
Você é um analista de dados executivo.
Analise os dados de vendas atualizados da planilha do Google Sheets abaixo e crie um relatório executivo estruturado.

DADOS DA PLANILHA EM TEMPO REAL:
{tabela_markdown}

ESTRUTURA DESEJADA DO RELATÓRIO:
- **Resumo Geral:** Total faturado e volume total de itens vendidos.
- **Destaques de Vendas:** Produtos e categorias líderes em receita.
- **Pontos de Atenção:** Oportunidades de melhoria ou concentração de vendas em poucos produtos.
- **Recomendações Práticas:** 2 a 3 ações recomendadas para a liderança.

Seja direto, profissional e use marcadores (bullet points).
"""

# 4. Enviar a solicitação com lógica de fallback para os modelos
modelos_tentativa = ["gemini-3.8-flash", "gemini-3.5-flash", "gemini-3.7-flash","gemini-3.5-flash-lite"]
relatorio = None

for modelo in modelos_tentativa:
    try:
        print(f"Solicitando análise com o modelo: {modelo}...")
        response = client.models.generate_content(
            model=modelo,
            contents=prompt,
        )
        relatorio = response.text
        print(f"Sucesso ao gerar com o modelo {modelo}!\n")
        break
    except Exception:
        print(f"O modelo {modelo} está em alta demanda. Tentando o próximo modelo...")

# 5. Exibir e guardar o relatório final
if relatorio:
    print("=== RELATÓRIO DA PLANILHA EM TEMPO REAL ===\n")
    print(relatorio)

    with open("relatorio_google_sheets.md", "w", encoding="utf-8") as f:
        f.write(relatorio)

    print("\nRelatório guardado com sucesso como 'relatorio_google_sheets.md'!")
else:
    print("Não foi possível conectar aos modelos neste momento. Tente executar a célula novamente.")

Solicitando análise com o modelo: gemini-3.8-flash...
O modelo gemini-3.8-flash está em alta demanda. Tentando o próximo modelo...
Solicitando análise com o modelo: gemini-3.5-flash...
Sucesso ao gerar com o modelo gemini-3.5-flash!

=== RELATÓRIO DA PLANILHA EM TEMPO REAL ===

Aqui está o relatório executivo de vendas estruturado com base na análise dos dados do período de 01/10/2026 a 06/10/2026.

---

# 📊 Relatório Executivo de Vendas

### 1. Resumo Geral
* **Faturamento Total:** R$ 43.400,00
* **Volume Total Vendido:** 56 unidades
* **Ticket Médio por Unidade:** R$ 775,00
* **Período Analisado:** 01 a 06 de outubro de 2026 (6 dias)

---

### 2. Destaques de Vendas
* **Categoria Líder em Receita:** **Eletrônicos** representou **82,5%** do faturamento total (R$ 35.800,00).
* **Produto Campeão em Faturamento:** **Notebook Pro** gerou R$ 27.000,00 (**62,2%** da receita total da empresa) com apenas 6 unidades vendidas.
* **Líder em Volume Físico:** **Mouse Sem Fio** foi o item mais vend

In [8]:
from google import genai
from google.colab import userdata

# 1. Carregar a chave
api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

# 2. Listar todos os modelos disponíveis na sua conta
print("Modelos disponíveis na sua chave:")
for m in client.models.list():
    print(m.name)

Modelos disponíveis na sua chave:
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/lyria-3-clip-preview
m